# 最优水桶问题

**类别：** 非线性

来源：[https://www.hexaly.com/templates/optimal-bucket-problem](https://www.hexaly.com/templates/optimal-bucket-problem)


## 问题描述

水桶的最佳形状是什么？在 **最优水桶问题** 中，我们希望设计一个能在不超过可用表面材料的情况下最大化所能容纳流体体积的水桶。一个水桶由三个值定义：底部圆盘的半径、顶部开口的半径以及高度，分别记为 r、R 和 h。问题在于选择 r、R 和 h 的值，以在水桶表面积不超过可用材料的约束下，最大化水桶的体积。

更多细节请参见 [DataGenetics](http://datagenetics.com/blog/january32015/index.html)。

	

### 建模要点

- 添加 [浮点决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#floating-point-decisions) 来建模水桶的尺寸
- 使用 [非线性算子](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#table-of-available-operators-and-functions) 来计算水桶的表面积和体积
- 了解 Hexaly Optimizer 的建模风格：[区分决策变量与中间表达式](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-decision-variables-from-intermediate-variableshttps://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-decision-variables-from-intermediate-variables)


## 模型

用于建造水桶的可用材料是一个半径为 1 的平面圆盘，其表面积为 S=π。在不超出该材料面积的前提下，我们尝试构造一个能容纳最大体积的水桶。

模型包含三个浮点决策变量。它们代表定义水桶形状的三个量：底部圆盘半径 r、顶部开口半径 R 以及高度 h。水桶的表面积和体积完全由这三个量确定，因此它们无需作为决策变量，而是作为中间表达式。水桶的表面积由表达式 S = π*r² + π(R+r)sqrt((R-r)²+h²) 给出。在计算之后，我们将其约束为不超过 π。然后我们可以计算水桶的体积，由表达式 V = (π*h)/3 * (R²+Rr+r²) 给出，并将其最大化。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

with hexaly.optimizer.HexalyOptimizer() as optimizer:
    PI = 3.14159265359

    #
    # Declare the optimization model
    #
    m = optimizer.model

    # Numerical decisions
    R = m.float(0, 1)
    r = m.float(0, 1)
    h = m.float(0, 1)

    # Surface must not exceed the surface of the plain disc
    surface = PI * r ** 2 + PI * (R + r) * m.sqrt((R - r) ** 2 + h ** 2)
    m.constraint(surface <= PI)

    # Maximize the volume
    volume = PI * h / 3 * (R ** 2 + R * r + r ** 2)
    m.maximize(volume)

    m.close()

    #
    # Parametrize the optimizer
    #
    if len(sys.argv) >= 3:
        optimizer.param.time_limit = int(sys.argv[2])
    else:
        optimizer.param.time_limit = 2

    optimizer.solve()

    #
    # Write the solution in a file with the following format:
    #  - surface and volume of the bucket
    #  - values of R, r and h
    #
    if len(sys.argv) >= 2:
        with open(sys.argv[1], 'w') as f:
            f.write("%f %f\n" % (surface.value, volume.value))
            f.write("%f %f %f\n" % (R.value, r.value, h.value))
